# 1.weaviate嵌入向量数据库示例

知识点讲解：Weaviate 向量数据库

1. 什么是 Weaviate
   - Weaviate 是完全用 Go 语言构建的开源云原生向量数据库，支持自托管与云服务（WCS）两种部署
   - 核心特点：GraphQL 原生查询、混合检索（向量+BM25）、强大的 schema 管理、内置多种向量化模块（OpenAI、Cohere、HuggingFace 等）
   - 与 Pinecone/TCVectorDB 一样有「集合」概念，集合类似关系型数据库的表，管理一类数据对象
   - 依赖安装：pip install langchain-weaviate weaviate-client

2. 部署方式与适用场景
   - Weaviate 云：官方托管，支持数据复制、零停机更新、无缝扩容；免费账号每实例最多 14 天，付费不限
   - Docker 部署：docker run 一条命令即可，暴露 8080(REST) 与 50051(gRPC) 端口，适合评估/开发
   - K8s 部署：适用于开发和生产
   - 嵌入式 Weaviate：基于本地文件，仅 Linux/macOS 支持，适合评估
   - 常用运维命令：docker start/stop/rm、docker ps / docker ps -a、docker images / docker rmi

3. 使用 Weaviate 的标准流程
   - 创建部署（云/Docker/K8s）→ 安装客户端与 LangChain 集成包 → 连接 Weaviate → 创建数据集/集合 → 添加数据/向量 → 相似性搜索（含过滤器）
   - 关键约束：集合名字必须以大写字母开头，且只能包含字母、数字、下划线，否则创建报错（与 Python 类名规范几乎一致）
   - 集合可在代码创建，也可在可视化管理界面（console.weaviate.cloud）创建；LangChain 会在使用时自动检测集合是否存在，不存在则直接创建

4. 连接方式
   - 本地连接 connect_to_local("host", "port")：用于 Docker 自托管实例，常用于开发测试
   - 云服务连接 connect_to_wcs(cluster_url=..., auth_credentials=AuthApiKey(...))：连接 WCS 托管实例，后台面板提供 REST 与 gRPC 两种地址及 API 密钥
   - 客户端对象 weaviate.WeaviateClient 封装了 REST/gRPC 连接，LangChain 组件底层复用该连接

5. Weaviate 核心概念
   - Class（类/集合）：相当于关系数据库的表，本例为 "Dataset"（注意大写开头命名）
   - Object（对象）：集合中的一条记录，含 id、properties、vector
   - Property（属性）：结构化数据字段，如 text、page、account_id
   - Schema（模式）：定义 Class 结构，包括属性类型、向量化配置、索引策略

6. 过滤查询的特点
   - 使用面向对象的 Filter API，而非字符串表达式或 JSON 字典
   - 链式调用：Filter.by_property("page").greater_or_equal(5)；.equal("pdf")；.contains_any(["ai","ml"])
   - 支持逻辑组合：& (and)、| (or)、~ (not)
     示例：(Filter.by_property("page") >= 5) & (Filter.by_property("type") == "pdf")

7. WeaviateVectorStore 关键参数
   - client：WeaviateClient 实例，封装连接信息
   - index_name：集合（Class）名称，相当于表名
   - text_key：原文在 properties 中的字段名，默认 "text"
   - embedding：嵌入模型，用于客户端向量化（也可配置 Weaviate 内置向量化模块）

8. Retriever 模式
   - as_retriever() 把 VectorStore 转换为 LangChain Retriever 接口
   - 统一返回 List[Document]（不含分数），适合直接接入 RAG 链
   - 默认调用 similarity_search，可通过 search_type 切换为 MMR 等策略

9. 适用场景与局限
   - 适用：需要混合检索（向量+关键词）的知识库、灵活 schema 与复杂元数据查询、需自托管又要完整数据库特性、多租户 SaaS（原生租户级物理隔离）
   - 局限：自托管运维成本高于 Serverless（Pinecone）；社区生态/文档完善度低于 Faiss、Pinecone；gRPC 在部分网络环境有兼容性问题

10. 最佳实践
    - 生产环境使用 WCS 或 K8s 部署，避免单机 Docker 的可靠性风险
    - 善用 Weaviate 内置向量化模块（如 text2vec-openai），可省去客户端嵌入调用
    - 复杂过滤条件应在 Weaviate 层完成，不要应用层后置过滤以保证性能
    - 定期备份 schema 定义，Class 删除后无法恢复
    - 更换嵌入模型需重建整个 Class


In [ ]:
import dotenv
import weaviate
from langchain_openai import OpenAIEmbeddings
from langchain_weaviate import WeaviateVectorStore
from weaviate.classes.query import Filter
dotenv.load_dotenv()
texts = [
    "笨笨是一只很喜欢睡觉的猫咪",
    "我喜欢在夜晚听音乐，这让我感到放松。",
    "猫咪在窗台上打盹，看起来非常可爱。",
    "学习新技能是每个人都应该追求的目标。",
    "我最喜欢的食物是意大利面，尤其是番茄酱的那种。",
    "昨晚我做了一个奇怪的梦，梦见自己在太空飞行。",
    "我的手机突然关机了，让我有些焦虑。",
    "阅读是我每天都会做的事情，我觉得很充实。",
    "他们一起计划了一次周末的野餐，希望天气能好。",
    "我的狗喜欢追逐球，看起来非常开心。",
]
metadatas = [
    {"page": 1},
    {"page": 2},
    {"page": 3},
    {"page": 4},
    {"page": 5},
    {"page": 6, "account_id": 1},
    {"page": 7},
    {"page": 8},
    {"page": 9},
    {"page": 10},
]
client = weaviate.connect_to_local("192.168.2.120", "8080")

In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = WeaviateVectorStore(
    client=client,
    index_name="Dataset",
    text_key="text",
    embedding=embedding,
)
ids = db.add_texts(texts, metadatas)
print(ids)
filters = Filter.by_property("page").greater_or_equal(5)
print(db.similarity_search_with_score("笨笨", filters=filters))
retriever = db.as_retriever()
print(retriever.invoke("笨笨"))